In [10]:
import pandas as pd


# Load clinical data
clinical_df = pd.read_csv(
    "../ucec_tcga_pan_can_atlas_2018/data_clinical_patient.txt",
    sep="\t",
    comment="#",
    low_memory=False
)
clinical_df = clinical_df.set_index('PATIENT_ID')

status_df = pd.read_csv(
    "../ucec_tcga_pan_can_atlas_2018/data_timeline_status.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

treatment_df = pd.read_csv(
    "../ucec_tcga_pan_can_atlas_2018/data_timeline_treatment.txt",
    sep="\t",
    comment="#",   # skip header comments if present
    low_memory=False
)

# 1) Robustly get the STATUS column even if name has weird case/whitespace
col = None
for c in status_df.columns:
    if c.strip().lower() == "status":
        col = c
        break
if col is None:
    raise KeyError("No column named 'STATUS' (case-insensitive) found in status_df. Columns: " + ", ".join(status_df.columns))

# 2) Print raw counts (exclude NaN by default)
print("Raw counts (excluding NaN):")
print(status_df[col].value_counts())

# 3) Print counts including NaN
print("\nCounts including NaN:")
print(status_df[col].value_counts(dropna=False))

# 4) Print proportions (fractions)
print("\nProportions:")
print(status_df[col].value_counts(normalize=True, dropna=False))

# Replace with the actual column name if it's different (e.g., "PATIENT_ID", "CASE_ID", "bcr_patient_barcode")
patient_col = None
for c in status_df.columns:
    if c.strip().lower() in ["patient_id"]:
        patient_col = c
        break

if patient_col is None:
    raise KeyError("No patient ID column found in status_df. Columns: " + ", ".join(status_df.columns))

unique_patient_count = status_df[patient_col].nunique()
print("Number of unique patients:", unique_patient_count)
print(clinical_df["DFS_STATUS"].value_counts(dropna=False))


Raw counts (excluding NaN):
STATUS
Initial Diagnosis          523
Last Follow Up             441
DECEASED                    87
Metastatic                  27
Locoregional Recurrence     24
Last Known Alive            11
New Primary Tumor           11
Locoregional Disease        11
Distant Metastasis          10
Name: count, dtype: int64

Counts including NaN:
STATUS
Initial Diagnosis          523
Last Follow Up             441
DECEASED                    87
Metastatic                  27
Locoregional Recurrence     24
Last Known Alive            11
New Primary Tumor           11
Locoregional Disease        11
Distant Metastasis          10
Name: count, dtype: int64

Proportions:
STATUS
Initial Diagnosis          0.456769
Last Follow Up             0.385153
DECEASED                   0.075983
Metastatic                 0.023581
Locoregional Recurrence    0.020961
Last Known Alive           0.009607
New Primary Tumor          0.009607
Locoregional Disease       0.009607
Distant Metastas

In [11]:
# import pandas as pd
# import numpy as np

# def label_patients(status_df, min_followup_days=1095):

#     # --- define recurrence-related event names ---
#     recurrence_events = {
#         "Locoregional Recurrence",
#         "Distant Metastasis",
#         "Metastatic"
#     }

#     # --- define events that disqualify no-recurrence patients ---
#     problematic_events = recurrence_events.union({
#         "New Primary Tumor",
#         "Locoregional Disease"
#     })

#     results = []

#     # group timeline by patient
#     for pid, df in status_df.groupby("PATIENT_ID"):
#         df = df.sort_values("START_DATE")

#         # -------------------------
#         # STEP 1: Identify recurrence
#         # -------------------------
#         recurrence_rows = df[df["STATUS"].isin(recurrence_events)]

#         if len(recurrence_rows) > 0:
#             first_rec = recurrence_rows.iloc[0]
#             results.append({
#                 "PATIENT_ID": pid,
#                 "LABEL": 1,
#                 "FOLLOW_UP_DAYS": first_rec["START_DATE"],
#                 "RECURRENCE_DATE": first_rec["START_DATE"],
#                 "RECURRENCE_TYPE": first_rec["STATUS"]
#             })
#             continue  # recurrence overrides everything else

#         # -------------------------
#         # STEP 2: Identify no recurrence
#         # -------------------------

#         # A. Must have no problematic events
#         if df["STATUS"].isin(problematic_events).any():
#             # can't be labeled no recurrence
#             results.append({
#                 "PATIENT_ID": pid,
#                 "LABEL": np.nan,
#                 "FOLLOW_UP_DAYS": None,
#                 "RECURRENCE_DATE": None,
#                 "RECURRENCE_TYPE": None
#             })
#             continue

#         # B. Must have a valid Last Follow Up event
#         lf_rows = df[df["STATUS"] == "Last Follow Up"]

#         if len(lf_rows) == 0:
#             # exclude Last Known Alive and DECEASED completely
#             results.append({
#                 "PATIENT_ID": pid,
#                 "LABEL": np.nan,
#                 "FOLLOW_UP_DAYS": None,
#                 "RECURRENCE_DATE": None,
#                 "RECURRENCE_TYPE": None
#             })
#             continue

#         # take the *latest* last follow-up
#         last_follow = lf_rows.sort_values("START_DATE").iloc[-1]
#         followup_days = last_follow["START_DATE"]

#         # Must satisfy follow-up >= 3 years
#         if followup_days < min_followup_days:
#             results.append({
#                 "PATIENT_ID": pid,
#                 "LABEL": np.nan,
#                 "FOLLOW_UP_DAYS": followup_days,
#                 "RECURRENCE_DATE": None,
#                 "RECURRENCE_TYPE": None
#             })
#             continue

#         # C. Must be tumor-free at last follow up
#         # First check PRIMARY_THERAPY_OUTCOME_SUCCESS
#         ptos = last_follow["PRIMARY_THERAPY_OUTCOME_SUCCESS"]

#         if pd.notna(ptos):
#             if ptos != "Complete Remission/Response":
#                 results.append({
#                     "PATIENT_ID": pid,
#                     "LABEL": np.nan,
#                     "FOLLOW_UP_DAYS": followup_days,
#                     "RECURRENCE_DATE": None,
#                     "RECURRENCE_TYPE": None
#                 })
#                 continue
#         else:
#             # If PTOS is NaN → check TUMOR_STATUS at last follow-up
#             tumor_status = last_follow["TUMOR_STATUS"]
#             if pd.isna(tumor_status) or tumor_status != "tumor_free":
#                 results.append({
#                     "PATIENT_ID": pid,
#                     "LABEL": np.nan,
#                     "FOLLOW_UP_DAYS": followup_days,
#                     "RECURRENCE_DATE": None,
#                     "RECURRENCE_TYPE": None
#                 })
#                 continue

#         # If we reached here → patient is no recurrence
#         results.append({
#             "PATIENT_ID": pid,
#             "LABEL": 0,
#             "FOLLOW_UP_DAYS": followup_days,
#             "RECURRENCE_DATE": None,
#             "RECURRENCE_TYPE": None
#         })

#     return pd.DataFrame(results)

# num_positive = (labels_df["LABEL"] == 1).sum()
# num_negative = (labels_df["LABEL"] == 0).sum()
# print(num_positive, num_negative)
###### OLD LABELING FUNCTION< NOT GOOD

In [12]:
import pandas as pd
import numpy as np

import pandas as pd
import numpy as np

def label_patients(status_df, treatment_df=None, clinical_df=None, min_followup_days=1095):
    """
    Label patients as recurrence (1), no recurrence (0), or unknown (NaN).

    Rules:
    1. Positive for recurrence (1) if:
       - STATUS in status_df contains any of:
         "Locoregional Recurrence", "Distant Metastasis", "Metastatic"
       - OR treatment_df contains:
         ANATOMIC_TREATMENT_SITE in ["Local Recurrence", "Distant Recurrence"]
         AND REGIMEN_INDICATION == "Recurrence"
       - Use START_DATE from the corresponding row as RECURRENCE_DATE.
    2. Verify DFS_STATUS in clinical_df:
       - If patient is labeled 1 but DFS_STATUS == "0:DiseaseFree", raise error.
       - If patient is labeled 0 but DFS_STATUS == "1:Recurred/Progressed", raise error.
    3. No recurrence (0) if:
       - STATUS does NOT contain:
         "Locoregional Recurrence", "Distant Metastasis", "Metastatic",
         "New Primary Tumor", "Locoregional Disease", "DECEASED"
       - Last Follow Up row exists with START_DATE >= min_followup_days
       - PRIMARY_THERAPY_OUTCOME_SUCCESS == "Complete Remission/Response"
         OR (NaN and TUMOR_STATUS == "tumor_free")
    4. Otherwise label as NaN.

    Returns a DataFrame with columns:
        PATIENT_ID, LABEL, FOLLOW_UP_DAYS, RECURRENCE_DATE, RECURRENCE_TYPE
    """
    recurrence_events = {"Locoregional Recurrence", "Distant Metastasis", "Metastatic"}
    problematic_events = recurrence_events.union({"New Primary Tumor", "Locoregional Disease", "DECEASED"})
    
    # For treatment_df recurrence labeling
    treatment_recur_sites = {"Local Recurrence", "Distant Recurrence"}
    
    results = []

    # Group by patient
    for pid, df in status_df.groupby("PATIENT_ID"):
        df = df.sort_values("START_DATE")
        
        # STEP 1: Recurrence from status_df
        recur_rows = df[df["STATUS"].isin(recurrence_events)]
        rec_date = None
        rec_type = None
        
        if len(recur_rows) > 0:
            first_recur = recur_rows.iloc[0]
            rec_date = first_recur["START_DATE"]
            rec_type = first_recur["STATUS"]
            results.append({
                "PATIENT_ID": pid,
                "LABEL": 1,
                "FOLLOW_UP_DAYS": rec_date,
                "RECURRENCE_DATE": rec_date,
                "RECURRENCE_TYPE": rec_type
            })
            continue
        
        # STEP 1b: Recurrence from treatment_df
        if treatment_df is not None:
            t_df = treatment_df[treatment_df["PATIENT_ID"] == pid]
        
            # Find first row with ANATOMIC_TREATMENT_SITE in recurrence sites
            site_recur = t_df[t_df["ANATOMIC_TREATMENT_SITE"].isin(treatment_recur_sites)]
            first_site_row = site_recur.sort_values("START_DATE").head(1)
        
            # Find first row with REGIMEN_INDICATION == "Recurrence"
            regimen_recur = t_df[t_df["REGIMEN_INDICATION"] == "Recurrence"]
            first_regimen_row = regimen_recur.sort_values("START_DATE").head(1)
        
            # Combine the two, take the earliest START_DATE
            combined = pd.concat([first_site_row, first_regimen_row])
            if len(combined) > 0:
                if pid == "TCGA-AP-A0LH":
                    print("HI")
                first_recur = combined.sort_values("START_DATE").iloc[0]
                rec_date = first_recur["START_DATE"]
                rec_type = f"Treatment {first_recur['ANATOMIC_TREATMENT_SITE']}"
                results.append({
                    "PATIENT_ID": pid,
                    "LABEL": 1,
                    "FOLLOW_UP_DAYS": rec_date,
                    "RECURRENCE_DATE": rec_date,
                    "RECURRENCE_TYPE": rec_type
                })
                if pid == "TCGA-AP-A0LH":
                    print("HI")
                continue
        
        # STEP 2: No recurrence checks
        if df["STATUS"].isin(problematic_events).any():
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": df["START_DATE"].max(),
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        
        # Must have Last Follow Up
        lf_rows = df[df["STATUS"] == "Last Follow Up"]
        if len(lf_rows) == 0:
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": df["START_DATE"].max(),
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        
        last_follow = lf_rows.iloc[-1]
        followup_days = last_follow["START_DATE"]
        if followup_days < min_followup_days:
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": followup_days,
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        
        # Check tumor-free status
        ptos = last_follow.get("PRIMARY_THERAPY_OUTCOME_SUCCESS", np.nan)
        tumor = last_follow.get("TUMOR_STATUS", np.nan)
        if pd.notna(ptos) and ptos != "Complete Remission/Response":
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": followup_days,
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        elif pd.isna(ptos) and (pd.isna(tumor) or str(tumor).lower() != "tumor_free"):
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": followup_days,
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        
        # STEP 3: Label no recurrence
        results.append({
            "PATIENT_ID": pid,
            "LABEL": 0,
            "FOLLOW_UP_DAYS": followup_days,
            "RECURRENCE_DATE": None,
            "RECURRENCE_TYPE": None
        })
    
    labels_df = pd.DataFrame(results)

    # STEP 4: Optional DFS_STATUS verification
    if clinical_df is not None:
        merged = labels_df.merge(clinical_df[["PATIENT_ID", "DFS_STATUS"]], on="PATIENT_ID", how="left")
        for _, row in merged.iterrows():
            if row["LABEL"] == 1 and row["DFS_STATUS"] == "0:DiseaseFree":
                # raise ValueError(f"Patient {row['PATIENT_ID']} labeled as recurrence but DFS_STATUS is DiseaseFree")
                print(f"Patient {row['PATIENT_ID']} labeled as recurrence but DFS_STATUS is DiseaseFree")
            if row["LABEL"] == 0 and row["DFS_STATUS"] == "1:Recurred/Progressed":
                # raise ValueError(f"Patient {row['PATIENT_ID']} labeled as no recurrence but DFS_STATUS is Recurred/Progressed")
                print(f"Patient {row['PATIENT_ID']} labeled as no recurrence but DFS_STATUS is Recurred/Progressed")

    return labels_df



def label_patients_simple(status_df, clinical_df=None, min_followup_days=1095):
    """
    Label patients as recurrence (1), no recurrence (0), or unknown (NaN).

    Lableing Algorithm:
    1. If a patient under DFS_STATUS is Recurred/Progressed, label 1.
    2. If a patient has a followup days greater than or equal to min_followup_days and DFS_STATUS is "0:DiseaseFree" lable 0
    3. If a patient does not have data for DFS_STATUS, but NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT is yes, label 1.
    3. If a patient does not have data for DFS_STATUS, but NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT is no
       and has a followup days greater than or equal to min_followup_days, label 0.
    4. Otherwise, lavel 


    
    1. Positive for recurrence (1) if:
       - STATUS in status_df contains any of:
         "Locoregional Recurrence", "Distant Metastasis", "Metastatic"
       - OR treatment_df contains:
         ANATOMIC_TREATMENT_SITE in ["Local Recurrence", "Distant Recurrence"]
         AND REGIMEN_INDICATION == "Recurrence"
       - Use START_DATE from the corresponding row as RECURRENCE_DATE.
    2. Verify DFS_STATUS in clinical_df:
       - If patient is labeled 1 but DFS_STATUS == "0:DiseaseFree", raise error.
       - If patient is labeled 0 but DFS_STATUS == "1:Recurred/Progressed", raise error.
    3. No recurrence (0) if:
       - STATUS does NOT contain:
         "Locoregional Recurrence", "Distant Metastasis", "Metastatic",
         "New Primary Tumor", "Locoregional Disease", "DECEASED"
       - Last Follow Up row exists with START_DATE >= min_followup_days
       - PRIMARY_THERAPY_OUTCOME_SUCCESS == "Complete Remission/Response"
         OR (NaN and TUMOR_STATUS == "tumor_free")
    4. Otherwise label as NaN.

    Returns a DataFrame with columns:
        PATIENT_ID, LABEL, FOLLOW_UP_DAYS, RECURRENCE_DATE, RECURRENCE_TYPE
    """
    recurrence_events = {"Locoregional Recurrence", "Distant Metastasis", "Metastatic"}
    problematic_events = recurrence_events.union({"New Primary Tumor", "Locoregional Disease", "DECEASED"})
    
    # For treatment_df recurrence labeling
    treatment_recur_sites = {"Local Recurrence", "Distant Recurrence"}
    
    results = []

    # Group by patient
    for pid, df in status_df.groupby("PATIENT_ID"):
        df = df.sort_values("START_DATE")
        
        # STEP 1: Recurrence from status_df
        recur_rows = df[df["STATUS"].isin(recurrence_events)]
        rec_date = None
        rec_type = None
        
        if len(recur_rows) > 0:
            first_recur = recur_rows.iloc[0]
            rec_date = first_recur["START_DATE"]
            rec_type = first_recur["STATUS"]
            results.append({
                "PATIENT_ID": pid,
                "LABEL": 1,
                "FOLLOW_UP_DAYS": rec_date,
                "RECURRENCE_DATE": rec_date,
                "RECURRENCE_TYPE": rec_type
            })
            continue
        
        # STEP 1b: Recurrence from treatment_df
        if treatment_df is not None:
            t_df = treatment_df[treatment_df["PATIENT_ID"] == pid]
        
            # Find first row with ANATOMIC_TREATMENT_SITE in recurrence sites
            site_recur = t_df[t_df["ANATOMIC_TREATMENT_SITE"].isin(treatment_recur_sites)]
            first_site_row = site_recur.sort_values("START_DATE").head(1)
        
            # Find first row with REGIMEN_INDICATION == "Recurrence"
            regimen_recur = t_df[t_df["REGIMEN_INDICATION"] == "Recurrence"]
            first_regimen_row = regimen_recur.sort_values("START_DATE").head(1)
        
            # Combine the two, take the earliest START_DATE
            combined = pd.concat([first_site_row, first_regimen_row])
            if len(combined) > 0:
                if pid == "TCGA-AP-A0LH":
                    print("HI")
                first_recur = combined.sort_values("START_DATE").iloc[0]
                rec_date = first_recur["START_DATE"]
                rec_type = f"Treatment {first_recur['ANATOMIC_TREATMENT_SITE']}"
                results.append({
                    "PATIENT_ID": pid,
                    "LABEL": 1,
                    "FOLLOW_UP_DAYS": rec_date,
                    "RECURRENCE_DATE": rec_date,
                    "RECURRENCE_TYPE": rec_type
                })
                if pid == "TCGA-AP-A0LH":
                    print("HI")
                continue
        
        # STEP 2: No recurrence checks
        if df["STATUS"].isin(problematic_events).any():
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": df["START_DATE"].max(),
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        
        # Must have Last Follow Up
        lf_rows = df[df["STATUS"] == "Last Follow Up"]
        if len(lf_rows) == 0:
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": df["START_DATE"].max(),
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        
        last_follow = lf_rows.iloc[-1]
        followup_days = last_follow["START_DATE"]
        if followup_days < min_followup_days:
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": followup_days,
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        
        # Check tumor-free status
        ptos = last_follow.get("PRIMARY_THERAPY_OUTCOME_SUCCESS", np.nan)
        tumor = last_follow.get("TUMOR_STATUS", np.nan)
        if pd.notna(ptos) and ptos != "Complete Remission/Response":
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": followup_days,
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        elif pd.isna(ptos) and (pd.isna(tumor) or str(tumor).lower() != "tumor_free"):
            results.append({
                "PATIENT_ID": pid,
                "LABEL": np.nan,
                "FOLLOW_UP_DAYS": followup_days,
                "RECURRENCE_DATE": None,
                "RECURRENCE_TYPE": None
            })
            continue
        
        # STEP 3: Label no recurrence
        results.append({
            "PATIENT_ID": pid,
            "LABEL": 0,
            "FOLLOW_UP_DAYS": followup_days,
            "RECURRENCE_DATE": None,
            "RECURRENCE_TYPE": None
        })
    
    labels_df = pd.DataFrame(results)

    # STEP 4: Optional DFS_STATUS verification
    if clinical_df is not None:
        merged = labels_df.merge(clinical_df[["PATIENT_ID", "DFS_STATUS"]], on="PATIENT_ID", how="left")
        for _, row in merged.iterrows():
            if row["LABEL"] == 1 and row["DFS_STATUS"] == "0:DiseaseFree":
                # raise ValueError(f"Patient {row['PATIENT_ID']} labeled as recurrence but DFS_STATUS is DiseaseFree")
                print(f"Patient {row['PATIENT_ID']} labeled as recurrence but DFS_STATUS is DiseaseFree")
            if row["LABEL"] == 0 and row["DFS_STATUS"] == "1:Recurred/Progressed":
                # raise ValueError(f"Patient {row['PATIENT_ID']} labeled as no recurrence but DFS_STATUS is Recurred/Progressed")
                print(f"Patient {row['PATIENT_ID']} labeled as no recurrence but DFS_STATUS is Recurred/Progressed")

    return labels_df





patient_labels = label_patients(status_df, treatment_df, clinical_df)
print(patient_labels["LABEL"].value_counts(dropna=False))

HI
HI


KeyError: "['PATIENT_ID'] not in index"

In [13]:
import pandas as pd

# All unique patient IDs across labels_df and clinical_df
all_patient_ids = pd.Index(labels_df.index if "PATIENT_ID" not in labels_df.columns else labels_df["PATIENT_ID"]).union(
    clinical_df.index if "PATIENT_ID" not in clinical_df.columns else clinical_df["PATIENT_ID"]
)

# LABELS series: NaN preserved, "MISSING" for patients not in labels_df
labels_series = labels_df["LABEL"] if "PATIENT_ID" not in labels_df.columns else labels_df.set_index("PATIENT_ID")["LABEL"]
labels_series = labels_series.reindex(all_patient_ids)  # add missing patient IDs
labels_series.loc[labels_series.index.difference(labels_df.index if "PATIENT_ID" not in labels_df.columns else labels_df["PATIENT_ID"])] = "MISSING"

# DFS_STATUS series: NaN preserved, "MISSING" for patients not in clinical_df
dfs_series = clinical_df["DFS_STATUS"] if "PATIENT_ID" not in clinical_df.columns else clinical_df.set_index("PATIENT_ID")["DFS_STATUS"]
dfs_series = dfs_series.reindex(all_patient_ids)
dfs_series.loc[dfs_series.index.difference(clinical_df.index if "PATIENT_ID" not in clinical_df.columns else clinical_df["PATIENT_ID"])] = "MISSING"

# Combine into a DataFrame
compare_df = pd.DataFrame({
    "LABEL": labels_series,
    "DFS_STATUS": dfs_series
})

# Print pair counts including NaNs and "MISSING"
pair_counts = compare_df.value_counts(dropna=False)
print(pair_counts)


NameError: name 'labels_df' is not defined

In [ ]:
def print_value_counts(df, column_name):
    if column_name not in df.columns:
        print(f"Column '{column_name}' not found in DataFrame.")
        return
    
    value_counts = df[column_name].value_counts(dropna=False)  # includes NaN
    for value, count in value_counts.items():
        print(f"{value}: {count}")

print_value_counts(data_timeline_treatment_df, "ANATOMIC_TREATMENT_SITE")
print()
print_value_counts(data_timeline_treatment_df, "REGIMEN_INDICATION")


In [ ]:
num_unique = data_timeline_treatment_df.iloc[:, 0].nunique()
print(f"Number of distinct values in first column: {num_unique}")


In [ ]:
def count_recurrence_rows(df):
    # Make everything lowercase string for safe matching
    df_str = df.astype(str).apply(lambda col: col.str.lower())

    # Check each cell for 'recurrence'
    recurrence_mask = df_str.apply(lambda col: col.str.contains("recurrence", na=False))

    # Count how many columns per row contain 'recurrence'
    recurrence_counts = recurrence_mask.sum(axis=1)

    # Total number of rows with at least one 'recurrence'
    total_rows_with_recurrence = (recurrence_counts > 0).sum()
    total_rows_with_2_recurrence = (recurrence_counts > 1).sum()

    print(f"Total rows with at least one 'recurrence': {total_rows_with_recurrence}")
    print(f"Total rows with at 2 'recurrence': {total_rows_with_2_recurrence}")
    return recurrence_counts

recurrence_counts = count_recurrence_rows(data_timeline_treatment_df)


In [ ]:
def count_recurrence_overlap(df, col1, col2):
    col1_contains = df[col1].astype(str).str.strip().str.lower().str.contains("recurrence", na=False)
    col2_contains = df[col2].astype(str).str.strip().str.lower().str.contains("recurrence", na=False)

    both = col1_contains & col2_contains
    only_col1 = col1_contains & ~col2_contains
    only_col2 = col2_contains & ~col1_contains

    print(f"Rows with recurrence in BOTH {col1} and {col2}: {both.sum()}")
    print(f"Rows with recurrence ONLY in {col1}: {only_col1.sum()}")
    print(f"Rows with recurrence ONLY in {col2}: {only_col2.sum()}")
    print(f"Rows with recurrence in EITHER column: {(both | only_col1 | only_col2).sum()}")

count_recurrence_overlap(data_timeline_treatment_df, "ANATOMIC_TREATMENT_SITE", "REGIMEN_INDICATION")


In [ ]:
# Just used for looking at the data for the labels
pair_counts = data_timeline_treatment_df.groupby(["ANATOMIC_TREATMENT_SITE", 'REGIMEN_INDICATION'], dropna=False).size().reset_index(name='Count')

# Print the pairings and the count
print(pair_counts)



In [ ]:
def count_unique_recurrence_patients(df, id_col="PATIENT_ID"):
    # Normalize strings
    df_str = df.astype(str).apply(lambda col: col.str.strip().str.lower())

    # Masks for recurrence conditions
    anat_mask = df_str["ANATOMIC_TREATMENT_SITE"].isin(["local recurrence", "distant recurrence"])
    regimen_mask = df_str["REGIMEN_INDICATION"] == "recurrence"

    # Get patient IDs
    anat_patients = set(df.loc[anat_mask, id_col])
    regimen_patients = set(df.loc[regimen_mask, id_col])

    # Union of both sets
    all_recurrence_patients = anat_patients | regimen_patients

    print(f"Unique patients with recurrence in ANATOMIC_TREATMENT_SITE: {len(anat_patients)}")
    print(f"Unique patients with recurrence in REGIMEN_INDICATION: {len(regimen_patients)}")
    print(f"Total unique patients with recurrence in either: {len(all_recurrence_patients)}")

    return anat_patients, regimen_patients, all_recurrence_patients


anat_patients, regimen_patients, all_recurrence_patients = count_unique_recurrence_patients(data_timeline_treatment_df)


In [ ]:
def get_recurrence_patients_from_timeline_treatment(filepath, id_col="PATIENT_ID"):
    # Load file
    df = pd.read_csv(filepath, sep="\t", comment="#", low_memory=False)

    # Masks for recurrence conditions
    anat_mask = df["ANATOMIC_TREATMENT_SITE"].isin(["Local Recurrence", "Distant Recurrence"])
    regimen_mask = df["REGIMEN_INDICATION"] == "Recurrence"
    
    # Collect patient IDs
    anat_patients = set(df.loc[anat_mask, id_col])
    regimen_patients = set(df.loc[regimen_mask, id_col])
    
    # Union of both
    all_patients = anat_patients | regimen_patients
    return all_patients

def get_locoregional_recurrence_patients(filepath, id_col=0, status_col="STATUS"):
    # Load file
    df = pd.read_csv(filepath, sep="\t", comment="#", low_memory=False)

    # Normalize STATUS values for safe matching
    mask = df[status_col].astype(str).str.strip().str.lower() == "locoregional recurrence"

    # Grab patient IDs from the first column (id_col=0 by default)
    patient_ids = df.iloc[mask.values, id_col].unique().tolist()

    return patient_ids

timeline_treatment_recur = get_recurrence_patients_from_timeline_treatment("../ucec_tcga_pan_can_atlas_2018/data_timeline_treatment.txt")
timeline_status_recur = get_locoregional_recurrence_patients("../ucec_tcga_pan_can_atlas_2018/data_timeline_status.txt")

In [ ]:
print(len(timeline_treatment_recur))
print(len(timeline_status_recur))

In [ ]:
# Combine both lists and remove duplicates by converting to a set, then back to a list
recur_patients = list(set(timeline_treatment_recur) | set(timeline_status_recur))
print(f"Number of unique recurrence patients: {len(recur_patients)}")

In [ ]:
import pandas as pd
import numpy as np

# Load clinical data
clinical_df = pd.read_csv(
    "../ucec_tcga_pan_can_atlas_2018/data_clinical_patient.txt",
    sep="\t",
    comment="#",
    low_memory=False
)
clinical_df = clinical_df.set_index('PATIENT_ID')

def assign_label(row):
    """
    Given a row, assigns:
    1 for recurrence,
    0 for no recurrence,
    None if recurrence information is missing.
    Uses NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT first, then DFS_STATUS if missing.
    """
    if row['NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT'] == 'Yes':
        return 1
    elif row['NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT'] == 'No':
        return 0
    elif pd.isna(row['NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT']):
        if row['DFS_STATUS'] == '1:Recurred/Progressed':
            return 1
        elif row['DFS_STATUS'] == '0:DiseaseFree':
            return 0
        else:
            return None

# Apply the labeling function
labels = clinical_df.apply(assign_label, axis=1)

# Get the patient IDs labeled as 1 (recurrence)
recurrence_patient_ids = labels[labels == 1].index.tolist()

print(f"Number of patients labeled as recurrence: {len(recurrence_patient_ids)}")


In [ ]:
# Convert both lists to sets for easy comparison
set_clinical = set(recurrence_patient_ids)
set_timeline = set(recur_patients)  # assuming recur_patients is your other list

# Overlap
overlap = set_clinical & set_timeline

# Unique to each
only_clinical = set_clinical - set_timeline
only_timeline = set_timeline - set_clinical

print(f"Number of patients in BOTH lists: {len(overlap)}")
print(f"Number of patients only in clinical_df list: {len(only_clinical)}")
print(f"Number of patients only in timeline list: {len(only_timeline)}")

# Optional: print example patient IDs
print(f"Example overlap IDs: {list(overlap)[:10]}")
print(f"Example only clinical IDs: {list(only_clinical)[:10]}")
print(f"Example only timeline IDs: {list(only_timeline)[:10]}")


In [ ]:
def compare_recurrence_labels(clinical_df, timeline_df, 
                              clinical_ids, timeline_ids,
                              clinical_id_col="PATIENT_ID",
                              status_col="NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT",
                              dfs_col="DFS_STATUS",
                              anat_col="ANATOMIC_TREATMENT_SITE",
                              regimen_col="REGIMEN_INDICATION"):
    """
    Compare clinical-based and timeline-based recurrence labels.
    
    Returns DataFrames of patients labeled only in one source and shows why.
    """
    # Convert lists to sets for comparison
    set_clinical = set(clinical_ids)
    set_timeline = set(timeline_ids)
    
    # Intersection and differences
    overlap = set_clinical & set_timeline
    only_clinical = set_clinical - set_timeline
    only_timeline = set_timeline - set_clinical
    
    print(f"Overlap patients: {len(overlap)}")
    print(f"Only in clinical labels: {len(only_clinical)}")
    print(f"Only in timeline labels: {len(only_timeline)}")
    
    # Examine why clinical-only patients were missing in timeline
    clinical_missing = timeline_df[timeline_df['PATIENT_ID'].isin(only_clinical)]
    
    # Examine why timeline-only patients were missing in clinical
    timeline_missing = clinical_df.loc[list(only_timeline), [status_col, dfs_col]]
    
    return {
        "overlap": overlap,
        "clinical_only": clinical_missing,
        "timeline_only": timeline_missing
    }

# Assuming:
# clinical_df has index PATIENT_ID and columns NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT, DFS_STATUS
# data_timeline_treatment_df is your timeline DataFrame
results = compare_recurrence_labels(
    clinical_df,
    data_timeline_treatment_df.reset_index(),  # need PATIENT_ID column, not index
    recurrence_patient_ids,
    recur_patients
)

print("Clinical-only patients (why missing in timeline):")
print(results["clinical_only"].head())

print("Timeline-only patients (why missing in clinical):")
print(results["timeline_only"].head())



In [ ]:
# Convert your patient ID lists to sets
set_clinical = set(recurrence_patient_ids)   # patients labeled as recurrence from clinical
set_timeline = set(recur_patients)           # patients labeled as recurrence from timeline

# Patients labeled as recurrence in both sources
overlap = set_clinical & set_timeline

# Patients labeled as recurrence in only one source
only_clinical = set_clinical - set_timeline
only_timeline = set_timeline - set_clinical

# Print summary
print(f"Patients labeled as recurrence in BOTH files: {len(overlap)}")
print(f"Patients labeled as recurrence ONLY in clinical file: {len(only_clinical)}")
print(f"Patients labeled as recurrence ONLY in timeline file: {len(only_timeline)}")
print(f"Total unique patients labeled as recurrence in at least one file: {len(overlap | only_clinical | only_timeline)}")


In [ ]:
# Convert patient ID lists to sets
set_clinical = set(recurrence_patient_ids)  # clinical recurrence labels
set_timeline = set(recur_patients)          # timeline recurrence labels

# Patients present in both datasets
patients_in_both = set(clinical_df.index) & set(data_timeline_treatment_df["PATIENT_ID"])

# Disagreements
clinical_only_disagree = (set_clinical - set_timeline) & patients_in_both
timeline_only_disagree = (set_timeline - set_clinical) & patients_in_both

print(f"Number of patients labeled recurrence in clinical but NOT in timeline: {len(clinical_only_disagree)}")
print(f"Number of patients labeled recurrence in timeline but NOT in clinical: {len(timeline_only_disagree)}")


In [ ]:
import pandas as pd

# Make sure your timeline DataFrame has PATIENT_ID as a column (not index)
timeline_df = data_timeline_treatment_df.copy()
if "PATIENT_ID" not in timeline_df.columns:
    timeline_df = timeline_df.reset_index()

# Columns to display
clinical_cols = ["NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT"]
timeline_cols = ["ANATOMIC_TREATMENT_SITE", "REGIMEN_INDICATION"]

def create_disagreement_table(patient_ids, clinical_df, timeline_df):
    """
    Returns a DataFrame for the given patient_ids showing relevant columns
    from both clinical and timeline data.
    """
    # Clinical data (some columns may be missing, handle with get)
    clinical_part = clinical_df.loc[list(patient_ids), clinical_cols].copy()
    
    # Timeline data (merge on PATIENT_ID)
    timeline_part = timeline_df[timeline_df["PATIENT_ID"].isin(patient_ids)][["PATIENT_ID"] + timeline_cols].copy()
    
    # Merge clinical and timeline info
    merged = clinical_part.reset_index().merge(timeline_part, left_on="PATIENT_ID", right_on="PATIENT_ID", how="outer")
    
    return merged

# Clinical-only disagreements
clinical_only_table = create_disagreement_table(list(clinical_only_disagree), clinical_df, timeline_df)
print("Patients labeled recurrence in clinical but not in timeline:")
print(clinical_only_table)

# Timeline-only disagreements
timeline_only_table = create_disagreement_table(timeline_only_disagree, clinical_df, timeline_df)
print("\nPatients labeled recurrence in timeline but not in clinical:")
print(timeline_only_table)


In [ ]:
import pandas as pd

# Make sure your timeline DataFrame has PATIENT_ID as a column (not index)
timeline_df = data_timeline_treatment_df.copy()
if "PATIENT_ID" not in timeline_df.columns:
    timeline_df = timeline_df.reset_index()

# Columns to display
clinical_cols = ["NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT"]
timeline_cols = ["ANATOMIC_TREATMENT_SITE", "REGIMEN_INDICATION"]

def create_disagreement_table(patient_ids, clinical_df, timeline_df):
    """
    Returns a DataFrame for the given patient_ids showing relevant columns
    from both clinical and timeline data.
    """
    # Clinical data (some columns may be missing, handle with get)
    clinical_part = clinical_df.loc[list(patient_ids), clinical_cols].copy()
    
    # Timeline data (merge on PATIENT_ID)
    timeline_part = timeline_df[timeline_df["PATIENT_ID"].isin(patient_ids)][["PATIENT_ID"] + timeline_cols].copy()
    
    # Merge clinical and timeline info
    merged = clinical_part.reset_index().merge(timeline_part, on="PATIENT_ID", how="outer")
    
    return merged

# Clinical-only disagreements
clinical_only_table = create_disagreement_table(list(clinical_only_disagree), clinical_df, timeline_df)
clinical_only_table.to_csv("clinical_only_disagreements.csv", index=False)
print("Saved clinical-only disagreements to 'clinical_only_disagreements.csv'")

# Timeline-only disagreements
timeline_only_table = create_disagreement_table(list(timeline_only_disagree), clinical_df, timeline_df)
timeline_only_table.to_csv("timeline_only_disagreements.csv", index=False)
print("Saved timeline-only disagreements to 'timeline_only_disagreements.csv'")


In [ ]:
import pandas as pd

# Ensure timeline DataFrame has PATIENT_ID as a column
timeline_df = data_timeline_treatment_df.copy()
if "PATIENT_ID" not in timeline_df.columns:
    timeline_df = timeline_df.reset_index()

# Columns to display
clinical_cols = ["NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT"]
timeline_cols = ["ANATOMIC_TREATMENT_SITE", "REGIMEN_INDICATION"]

def create_disagreement_table(patient_ids, clinical_df, timeline_df):
    """Return a merged, sorted, human-readable DataFrame for given patient_ids."""
    patient_ids = list(patient_ids)
    
    # Clinical data
    clinical_part = clinical_df.loc[patient_ids, clinical_cols].copy()
    
    # Timeline data
    timeline_part = timeline_df[timeline_df["PATIENT_ID"].isin(patient_ids)][["PATIENT_ID"] + timeline_cols].copy()
    
    # Merge
    merged = clinical_part.reset_index().merge(timeline_part, on="PATIENT_ID", how="outer")
    
    # Reorder and rename columns for readability
    merged = merged[["PATIENT_ID"] + clinical_cols + timeline_cols]
    merged = merged.rename(columns={
        "NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT": "Clinical_New_Tumor_Event",
        "ANATOMIC_TREATMENT_SITE": "Timeline_Anatomic_Treatment_Site",
        "REGIMEN_INDICATION": "Timeline_Regimen_Indication"
                })
    
    # Sort by PATIENT_ID
    merged = merged.sort_values("PATIENT_ID").reset_index(drop=True)
    
    return merged

def save_disagreement_table_text(df, filename):
    """Save a DataFrame to a nicely formatted text file."""
    with open(filename, "w") as f:
        f.write(df.to_string(index=False))
    print(f"Saved nicely formatted table to '{filename}'")

# Clinical-only disagreements
clinical_only_table = create_disagreement_table(list(clinical_only_disagree), clinical_df, timeline_df)
save_disagreement_table_text(clinical_only_table, "clinical_only_disagreements.txt")

# Timeline-only disagreements
timeline_only_table = create_disagreement_table(list(timeline_only_disagree), clinical_df, timeline_df)
save_disagreement_table_text(timeline_only_table, "timeline_only_disagreements.txt")


In [ ]:
import pandas as pd

def create_full_label_table(clinical_df, timeline_df, status_df, labels_df):
    """
    Create a merged table for all patients with recurrence info, status, and DFS months.
    Handles cases where PATIENT_ID is the index instead of a column.
    """
    # Ensure PATIENT_ID is a column in all dataframes
    for df_name, df in [("clinical_df", clinical_df), ("timeline_df", timeline_df), ("status_df", status_df)]:
        if "PATIENT_ID" not in df.columns:
            print(f"Note: 'PATIENT_ID' not found in columns of {df_name}. Using index instead.")
            df.reset_index(inplace=True)

    # Verify available columns
    expected_clinical_cols = ["PATIENT_ID", "DFS_STATUS", "DFS_MONTHS", "NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT"]
    expected_timeline_cols = ["PATIENT_ID", "ANATOMIC_TREATMENT_SITE", "REGIMEN_INDICATION"]

    missing_clinical = [c for c in expected_clinical_cols if c not in clinical_df.columns]
    missing_timeline = [c for c in expected_timeline_cols if c not in timeline_df.columns]

    if missing_clinical:
        print(f"Warning: Missing columns in clinical_df: {missing_clinical}")
    if missing_timeline:
        print(f"Warning: Missing columns in timeline_df: {missing_timeline}")

    # Select only the columns that exist
    clinical_part = clinical_df[[c for c in expected_clinical_cols if c in clinical_df.columns]].copy()
    timeline_part = timeline_df[[c for c in expected_timeline_cols if c in timeline_df.columns]].copy()

    # Merge clinical and timeline info
    merged = clinical_part.merge(timeline_part, on="PATIENT_ID", how="outer")

    # Add STATUS column if present
    if "STATUS" in status_df.columns:
        merged = merged.merge(status_df[["PATIENT_ID", "STATUS"]], on="PATIENT_ID", how="left")
    else:
        print("Warning: STATUS column not found in status_df.")

    labels_part = labels_df[[c for c in ["PATIENT_ID", "LABEL", "FOLLOW_UP_DAYS"] if c in labels_df.columns]].copy()
    merged = merged.merge(labels_part, on="PATIENT_ID", how="outer")
    
    # Sort by patient ID
    merged = merged.sort_values("PATIENT_ID").reset_index(drop=True)

    # Rename for clarity
    merged = merged.rename(columns={
        "NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT": "New_Tumor",
        "DFS_STATUS": "DFS_Status",
        "DFS_MONTHS": "DFS_Months",
        "ANATOMIC_TREATMENT_SITE": "Anatomic_Treatment",
        "REGIMEN_INDICATION": "Regimen_Indication"
    })

    return merged


def save_table_as_text(df, filename):
    """Save DataFrame as nicely formatted text file."""
    with open(filename, "w") as f:
        f.write(df.to_string(index=False))
    print(f"Saved full patient table to '{filename}'")


# Load status data
status_df = pd.read_csv(
    "../ucec_tcga_pan_can_atlas_2018/data_timeline_status.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

# Load status data
timeline_df = pd.read_csv(
    "../ucec_tcga_pan_can_atlas_2018/data_timeline_treatment.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

print(labels_df.columns)
# Create the full table
full_table = create_full_label_table(
    clinical_df=clinical_df,
    timeline_df=timeline_df,
    status_df=status_df,
    labels_df=labels_df
)

# Save as a text file
save_table_as_text(full_table, "labeling_columns.txt")


In [ ]:
# import pandas as pd

# # Ensure timeline DataFrame has PATIENT_ID as a column
# timeline_df = data_timeline_treatment_df.copy()
# if "PATIENT_ID" not in timeline_df.columns:
#     timeline_df = timeline_df.reset_index()

# # Columns to display
# clinical_cols = ["NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT", "DFS_STATUS"]
# timeline_cols = ["ANATOMIC_TREATMENT_SITE", "REGIMEN_INDICATION"]

# def create_full_label_table(clinical_df, timeline_df):
#     """
#     Create a merged table for all patients with recurrence info and disagreement markers.
#     """
#     # Merge clinical and timeline info
#     clinical_part = clinical_df[clinical_cols].copy().reset_index()
#     merged = clinical_part.merge(timeline_df[["PATIENT_ID"] + timeline_cols], on="PATIENT_ID", how="outer")
    
    
#     # Sort by patient ID
#     merged = merged.sort_values("PATIENT_ID").reset_index(drop=True)
    
#     # Rename columns for readability
#     merged = merged.rename(columns={
#         "NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT": "Clinical_New_Tumor_Event",
#         "DFS_STATUS": "Clinical_DFS_Status",
#         "ANATOMIC_TREATMENT_SITE": "Timeline_Anatomic_Treatment_Site",
#         "REGIMEN_INDICATION": "Timeline_Regimen_Indication"
#     })
    
#     return merged

# def save_table_as_text(df, filename):
#     """Save DataFrame as nicely formatted text file."""
#     with open(filename, "w") as f:
#         f.write(df.to_string(index=False))
#     print(f"Saved full patient table to '{filename}'")

# # Create the full table
# full_table = create_full_label_table(
#     clinical_df,
#     timeline_df,
# )

# # Save as a text file
# save_table_as_text(full_table, "all_patients_labels.txt")


In [ ]:
import pandas as pd
import numpy as np

def generate_recurrence_labels(treatment_file, status_file, clinical_file):
    """
    Generates a pd.Series of recurrence labels for all patients.
    
    Label rules:
     1 (recurred): 
        * ANATOMIC_TREATMENT_SITE = "Local Recurrence" or "Distant Recurrence"
        * REGIMEN_INDICATION = "Recurrence"
        * STATUS = "Locoregional Recurrence"
        * NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT = "Yes"
     0 (no recurrence): NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT = "No" (and no other columns show recurrence)
     None (unknown): all other patients
    """
    
    # --- Load data ---
    df_treatment = pd.read_csv(treatment_file, sep="\t", comment="#", low_memory=False)
    df_status = pd.read_csv(status_file, sep="\t", comment="#", low_memory=False)
    df_clinical = pd.read_csv(clinical_file, sep="\t", comment="#", low_memory=False)
    
    # Ensure PATIENT_ID is a column
    if df_treatment.index.name == "PATIENT_ID":
        df_treatment = df_treatment.reset_index()
    if df_clinical.index.name == "PATIENT_ID":
        df_clinical = df_clinical.reset_index()
    
    # --- Set of patient IDs labeled as recurrence ---
    recur_patients = set()
    
    # From treatment file
    treatment_mask = df_treatment["ANATOMIC_TREATMENT_SITE"].isin(["Local Recurrence", "Distant Recurrence"])
    regimen_mask = df_treatment["REGIMEN_INDICATION"] == "Recurrence"
    recur_patients.update(df_treatment.loc[treatment_mask | regimen_mask, "PATIENT_ID"])
    
    # From status file
    status_mask = df_status["STATUS"].astype(str).str.strip() == "Locoregional Recurrence"
    recur_patients.update(df_status.loc[status_mask, "PATIENT_ID"])
    
    # From clinical file
    clinical_yes_mask = df_clinical["NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT"].astype(str).str.strip().str.lower() == "yes"
    recur_patients.update(df_clinical.loc[clinical_yes_mask, "PATIENT_ID"])
    
    # --- Set of patients labeled as no recurrence ---
    clinical_no_mask = df_clinical["NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT"].astype(str).str.strip().str.lower() == "no"
    no_recur_patients = set(df_clinical.loc[clinical_no_mask, "PATIENT_ID"])
    
    # --- Combine all patient IDs ---
    all_patients = set(df_clinical["PATIENT_ID"]) | set(df_treatment["PATIENT_ID"]) | set(df_status["PATIENT_ID"])
    
    # --- Assign labels ---
    labels = {}
    for pid in all_patients:
        if pid in recur_patients:
            labels[pid] = 1
        elif pid in no_recur_patients:
            labels[pid] = 0
        else:
            labels[pid] = None
    
    # Return as pd.Series
    label_series = pd.Series(labels, name="Recurrence_Label")
    label_series.index.name = "PATIENT_ID"
    
    return label_series


In [ ]:
labels = generate_recurrence_labels(
    treatment_file="../ucec_tcga_pan_can_atlas_2018/data_timeline_treatment.txt",
    status_file="../ucec_tcga_pan_can_atlas_2018/data_timeline_status.txt",
    clinical_file="../ucec_tcga_pan_can_atlas_2018/data_clinical_patient.txt"
)

# Drop patients with unknown labels if needed
labels_clean = labels[labels != None]

print(labels.value_counts())
print(labels)
